# 태양광 후보지 Rule-based 1차 검토

현재 후보 CSV의 원본 컬럼과 ML Feature 이름을 그대로 유지하면서 규칙 판정 컬럼만 추가합니다.

데이터프레임 이름은 다음으로 고정합니다.

- `test_df`: 원본 후보 데이터
- `rules_df`: 실행할 Rule 데이터
- `rule_reviewed_df`: Rule 판정 컬럼이 추가된 전체 데이터
- `rule_passed_df`: 다음 단계인 Vision AI 또는 ML 랭킹으로 전달할 데이터
- `rule_audit_df`: 후보별 규칙 적용 내역


In [ ]:
!pip -q install openpyxl


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

print("라이브러리 로드 완료")


## 1. 파일 경로와 실행 대상 설정


In [ ]:
# ============================================================
# 실행 대상을 LAND 또는 BUILDING으로 선택
# ============================================================
DATASET_TYPE = "land"
# DATASET_TYPE = "building"

RULE_XLSX_FILENAME = "태양광_RuleBase_실행용_수정본.xlsx"

LAND_TEST_FILENAME = "Land_Test_Chungcheong_Uninstalled(4).csv"
BUILDING_TEST_FILENAME = "Building_Test_Chungcheong_Uninstalled(2).csv"

BASE_DIR = Path("/content")
RULE_XLSX_PATH = BASE_DIR / RULE_XLSX_FILENAME

if DATASET_TYPE == "land":
    TEST_FILENAME = LAND_TEST_FILENAME
    OUTPUT_PREFIX = "Land"
elif DATASET_TYPE == "building":
    TEST_FILENAME = BUILDING_TEST_FILENAME
    OUTPUT_PREFIX = "Building"
else:
    raise ValueError("DATASET_TYPE은 'land' 또는 'building'이어야 합니다.")

TEST_PATH = BASE_DIR / TEST_FILENAME
OUTPUT_DIR = BASE_DIR / "rulebase_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 현재 단계에서는 EXCLUDE만 다음 단계에서 제거합니다.
# LEGAL_REVIEW는 확정 탈락이 아니므로 유지합니다.
NEXT_STEP_EXCLUDE_DECISIONS = {
    "EXCLUDE",
}

print("데이터 유형:", DATASET_TYPE)
print("입력 파일:", TEST_PATH)
print("Rule 파일:", RULE_XLSX_PATH)
print("결과 폴더:", OUTPUT_DIR)


## 2. 데이터 로드 및 원본 컬럼 확인


In [ ]:
if not RULE_XLSX_PATH.exists():
    raise FileNotFoundError(f"Rule 엑셀을 찾을 수 없습니다: {RULE_XLSX_PATH}")

if not TEST_PATH.exists():
    raise FileNotFoundError(f"후보 CSV를 찾을 수 없습니다: {TEST_PATH}")

test_df = pd.read_csv(TEST_PATH, low_memory=False)
rules_df = pd.read_excel(
    RULE_XLSX_PATH,
    sheet_name="Rule_Data",
)

# Rule 엑셀의 TRUE/FALSE가 문자열로 읽히는 경우까지 처리
rules_df["enabled"] = (
    rules_df["enabled"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes", "y"])
)

rules_df = rules_df[
    rules_df["enabled"]
].copy()

print("test_df 크기:", test_df.shape)
print("실행 Rule 수:", len(rules_df))
print("\\n원본 컬럼:")
print(test_df.columns.tolist())


## 3. Rule Engine용 보조 컬럼 생성


In [ ]:
# 원본 34개 컬럼은 변경하거나 삭제하지 않습니다.
# 아래 보조 컬럼만 추가합니다.

CHUNGNAM = [
    "천안시", "공주시", "보령시", "아산시", "서산시",
    "논산시", "계룡시", "당진시", "금산군", "부여군",
    "서천군", "청양군", "홍성군", "예산군", "태안군",
]

CHUNGBUK = [
    "청주시", "충주시", "제천시", "보은군", "옥천군",
    "영동군", "증평군", "진천군", "괴산군", "음성군",
    "단양군",
]

DAEJEON = [
    "동구", "중구", "서구", "유성구", "대덕구",
]

KNOWN_JURISDICTIONS = CHUNGNAM + CHUNGBUK


def normalize_jurisdiction(row):
    sido = str(row.get("시도", "") or "").strip()
    sigungu = str(row.get("시군구", "") or "").strip()
    address = str(row.get("address_ml", "") or "").strip()

    # 대전은 Rule 엑셀과 동일하게 "대전 유성구" 형식
    if "대전" in sido or address.startswith("대전"):
        for district in DAEJEON:
            if district in address or sigungu == district:
                return f"대전 {district}"

    if "세종" in sido or address.startswith("세종"):
        return "세종특별자치시"

    # 주소에 포함된 시군을 최우선 사용
    for name in KNOWN_JURISDICTIONS:
        if name in address:
            return name

    if sigungu in KNOWN_JURISDICTIONS:
        return sigungu

    # 주소에서 첫 시/군 명칭 재추출
    match = re.search(r"([가-힣]+(?:시|군))", address)
    if match:
        candidate = match.group(1)
        if candidate in KNOWN_JURISDICTIONS:
            return candidate

    return np.nan


def normalize_asset_type(value):
    text = str(value or "").strip().upper()

    if text in {"토지", "토지형", "LAND", "0"}:
        return "LAND"

    if text in {"건물", "건물형", "BUILDING", "1"}:
        return "BUILDING"

    return "UNKNOWN"


def normalize_installation_type(row):
    raw = str(row.get("설치구분", "") or "").strip().upper()
    asset_type = row["asset_type_norm"]

    if raw in {
        "GROUND",
        "ROOFTOP",
        "PARKING_CANOPY",
        "BIPV",
    }:
        return raw

    if asset_type == "LAND":
        return "GROUND"

    if asset_type == "BUILDING":
        return "ROOFTOP"

    return "UNKNOWN"


test_df = test_df.copy()

test_df["candidate_id"] = (
    test_df["source_id_ml"]
    .astype(str)
)

test_df["jurisdiction_norm"] = (
    test_df.apply(
        normalize_jurisdiction,
        axis=1,
    )
)

test_df["asset_type_norm"] = (
    test_df["자산구분_ML"]
    .map(normalize_asset_type)
)

test_df["installation_type_norm"] = (
    test_df.apply(
        normalize_installation_type,
        axis=1,
    )
)

# 현재 Rule에서 직접 사용하는 숫자형 컬럼
for column in [
    "slope_avg",
]:
    if column in test_df.columns:
        test_df[column] = pd.to_numeric(
            test_df[column],
            errors="coerce",
        )

display(
    test_df[
        [
            "source_id_ml",
            "address_ml",
            "시도",
            "시군구",
            "jurisdiction_norm",
            "자산구분_ML",
            "asset_type_norm",
            "installation_type_norm",
            "slope_avg",
        ]
    ].head(10)
)


## 4. 규칙 비교 함수


In [ ]:
def is_missing(value):
    if value is None:
        return True

    try:
        return bool(pd.isna(value))
    except Exception:
        return False


def parse_value(value):
    if is_missing(value):
        return None

    if isinstance(value, (bool, int, float)):
        return value

    text = str(value).strip()
    lower = text.lower()

    if lower == "true":
        return True

    if lower == "false":
        return False

    try:
        number = float(text)
        if number.is_integer():
            return int(number)
        return number
    except ValueError:
        return text


def compare_value(
    value,
    operator,
    threshold,
):
    operator = str(operator or "").strip().upper()

    if operator == "IS_NULL":
        return is_missing(value)

    if operator == "NOT_NULL":
        return not is_missing(value)

    if is_missing(value):
        return False

    threshold = parse_value(threshold)

    if operator == "EQ":
        return (
            value == threshold
            or str(value) == str(threshold)
        )

    if operator == "NE":
        return not (
            value == threshold
            or str(value) == str(threshold)
        )

    if operator in {
        "LT",
        "LTE",
        "GT",
        "GTE",
    }:
        try:
            left = float(value)
            right = float(threshold)
        except (TypeError, ValueError):
            return False

        if operator == "LT":
            return left < right
        if operator == "LTE":
            return left <= right
        if operator == "GT":
            return left > right
        if operator == "GTE":
            return left >= right

    if operator in {
        "IN",
        "NOT_IN",
    }:
        if isinstance(threshold, str):
            options = [
                item.strip()
                for item in threshold.split("|")
                if item.strip()
            ]
        else:
            options = [threshold]

        result = str(value) in {
            str(item)
            for item in options
        }

        if operator == "IN":
            return result

        return not result

    raise ValueError(
        f"지원하지 않는 operator: {operator}"
    )


## 5. 후보지별 Rule 적용


In [ ]:
DECISION_PRIORITY = {
    "EXCLUDE": 100,
    "EXCLUDE_CANDIDATE": 90,
    "LEGAL_REVIEW": 70,
    "UNKNOWN": 60,
    "PASS_EXCEPTION": 30,
    "PASS": 10,
    "NO_APPLICABLE_RULE": 0,
}


def scope_matches(row, rule):
    jurisdiction = str(
        rule.get("jurisdiction", "ALL")
        or "ALL"
    ).strip()

    asset_type = str(
        rule.get("asset_type", "ALL")
        or "ALL"
    ).strip()

    installation_type = str(
        rule.get("installation_type", "ALL")
        or "ALL"
    ).strip()

    if (
        jurisdiction != "ALL"
        and row["jurisdiction_norm"]
        != jurisdiction
    ):
        return False

    if (
        asset_type != "ALL"
        and row["asset_type_norm"]
        != asset_type
    ):
        return False

    if (
        installation_type != "ALL"
        and row["installation_type_norm"]
        != installation_type
    ):
        return False

    return True


def evaluate_rule(row, rule):
    required_columns = [
        col.strip()
        for col in str(
            rule.get(
                "required_columns",
                "",
            )
            or ""
        ).split(",")
        if col.strip()
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in row.index
    ]

    missing_values = [
        col
        for col in required_columns
        if (
            col in row.index
            and is_missing(row.get(col))
        )
    ]

    if missing_columns or missing_values:
        return {
            "evaluation_state": "NOT_EVALUABLE",
            "matched": False,
            "missing_columns": missing_columns,
            "missing_values": missing_values,
        }

    target_feature = str(
        rule["target_feature"]
    ).strip()

    target_matched = compare_value(
        row.get(target_feature),
        rule["operator"],
        rule.get("threshold_value"),
    )

    condition_feature = str(
        rule.get(
            "condition_feature",
            "",
        )
        or ""
    ).strip()

    if condition_feature:
        condition_matched = compare_value(
            row.get(condition_feature),
            rule.get("condition_operator"),
            rule.get("condition_value"),
        )
    else:
        condition_matched = True

    return {
        "evaluation_state": "EVALUATED",
        "matched": bool(
            target_matched
            and condition_matched
        ),
        "missing_columns": [],
        "missing_values": [],
    }


review_rows = []
audit_rows = []

for _, candidate in test_df.iterrows():
    matched_rules = []
    not_evaluable_rules = []

    applicable_rule_count = 0
    evaluated_rule_count = 0
    substantive_rule_count = 0
    substantive_evaluated_count = 0

    for _, rule in rules_df.iterrows():
        if not scope_matches(
            candidate,
            rule,
        ):
            continue

        applicable_rule_count += 1

        if (
            str(rule.get("rule_type", "")).upper()
            != "DATA_QUALITY"
        ):
            substantive_rule_count += 1

        evaluation = evaluate_rule(
            candidate,
            rule,
        )

        audit_rows.append({
            "source_id_ml": candidate["source_id_ml"],
            "address_ml": candidate["address_ml"],
            "jurisdiction_norm": candidate["jurisdiction_norm"],
            "asset_type_norm": candidate["asset_type_norm"],
            "rule_id": rule["rule_id"],
            "rule_type": rule["rule_type"],
            "evaluation_state": evaluation["evaluation_state"],
            "matched": evaluation["matched"],
            "decision": (
                rule["decision"]
                if evaluation["matched"]
                else ""
            ),
            "message": (
                rule["message"]
                if evaluation["matched"]
                else ""
            ),
            "missing_columns": "|".join(
                evaluation["missing_columns"]
            ),
            "missing_values": "|".join(
                evaluation["missing_values"]
            ),
        })

        if (
            evaluation["evaluation_state"]
            == "NOT_EVALUABLE"
        ):
            not_evaluable_rules.append(
                str(rule["rule_id"])
            )
            continue

        evaluated_rule_count += 1

        if (
            str(rule.get("rule_type", "")).upper()
            != "DATA_QUALITY"
        ):
            substantive_evaluated_count += 1

        if evaluation["matched"]:
            matched_rules.append({
                "rule_id": str(rule["rule_id"]),
                "rule_type": str(rule["rule_type"]),
                "decision": str(rule["decision"]),
                "severity": int(
                    rule.get("severity", 0)
                    or 0
                ),
                "message": str(
                    rule.get("message", "")
                    or ""
                ),
            })

    if matched_rules:
        matched_rules.sort(
            key=lambda item: (
                DECISION_PRIORITY.get(
                    item["decision"],
                    0,
                ),
                item["severity"],
            ),
            reverse=True,
        )

        final_decision = (
            matched_rules[0]["decision"]
        )

        final_message = (
            matched_rules[0]["message"]
        )

    elif substantive_rule_count == 0:
        final_decision = (
            "NO_APPLICABLE_RULE"
        )
        final_message = (
            "현재 데이터와 자산 유형에 "
            "적용 가능한 실질 규칙이 없음"
        )

    elif substantive_evaluated_count == 0:
        final_decision = "UNKNOWN"
        final_message = (
            "적용 대상 규칙은 있으나 "
            "필요한 데이터가 없어 평가 불가"
        )

    else:
        final_decision = "PASS"
        final_message = (
            "실행된 규칙 중 "
            "위반·검토 조건 없음"
        )

    reviewed = candidate.to_dict()

    reviewed.update({
        "Rule_Final_Decision": final_decision,
        "Rule_Final_Message": final_message,
        "Rule_Matched_Count": len(
            matched_rules
        ),
        "Rule_Matched_IDs": "|".join(
            item["rule_id"]
            for item in matched_rules
        ),
        "Rule_Matched_Messages": " | ".join(
            item["message"]
            for item in matched_rules
        ),
        "Rule_Applicable_Count": (
            applicable_rule_count
        ),
        "Rule_Evaluated_Count": (
            evaluated_rule_count
        ),
        "Rule_Not_Evaluable_Count": len(
            not_evaluable_rules
        ),
        "Rule_Not_Evaluable_IDs": "|".join(
            not_evaluable_rules
        ),
    })

    reviewed["Rule_Pass_For_Next_Step"] = (
        final_decision
        not in NEXT_STEP_EXCLUDE_DECISIONS
    )

    review_rows.append(reviewed)


rule_reviewed_df = pd.DataFrame(
    review_rows
)

rule_audit_df = pd.DataFrame(
    audit_rows
)

rule_passed_df = (
    rule_reviewed_df[
        rule_reviewed_df[
            "Rule_Pass_For_Next_Step"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

print("전체 후보:", len(rule_reviewed_df))
print("다음 단계 전달:", len(rule_passed_df))
print("\\n최종 판정 분포:")
display(
    rule_reviewed_df[
        "Rule_Final_Decision"
    ]
    .value_counts(dropna=False)
    .rename_axis("Rule_Final_Decision")
    .reset_index(name="count")
)


## 6. 결과 저장


In [ ]:
# 전체 Rule 검토 결과
reviewed_output_path = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_Test_Chungcheong_RuleReviewed.csv"
)

# Vision AI 또는 ML 랭킹에 전달할 후보
passed_output_path = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_Test_Chungcheong_RulePassed.csv"
)

# 후보별 Rule 적용 감사 로그
audit_output_path = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_Test_Chungcheong_RuleAudit.csv"
)

rule_reviewed_df.to_csv(
    reviewed_output_path,
    index=False,
    encoding="utf-8-sig",
)

rule_passed_df.to_csv(
    passed_output_path,
    index=False,
    encoding="utf-8-sig",
)

rule_audit_df.to_csv(
    audit_output_path,
    index=False,
    encoding="utf-8-sig",
)

print("전체 검토 결과:", reviewed_output_path)
print("다음 단계 전달:", passed_output_path)
print("Rule 감사 로그:", audit_output_path)

# 랭킹 노트북에 그대로 넣어도 모델 Feature 이름은 유지됨
original_feature_columns = [
    "ghi_avg_daily",
    "pvout_avg_daily",
    "dni_avg_daily",
    "dif_avg_daily",
    "gti_avg_daily",
    "temp_avg",
    "wind_speed_10m",
    "wind_speed_50m",
    "wind_speed_100m",
    "slope_avg",
    "slope_dir",
    "elevation_avg",
    "Hillshade",
    "Southness",
    "distance_to_substation_km",
    "distance_to_powerline_km",
    "substation_count_5km",
    "powerline_length_5km_km",
    "high_voltage_line_nearby_5km",
    "substation_max_voltage_kv",
    "powerline_max_voltage_kv",
    "substation_max_voltage_kv_missing",
    "powerline_max_voltage_kv_missing",
    "asset_type_code",
]

missing_after_rule = [
    col
    for col in original_feature_columns
    if col not in rule_passed_df.columns
]

if missing_after_rule:
    raise KeyError(
        "Rule 처리 후 ML Feature가 사라졌습니다: "
        f"{missing_after_rule}"
    )

print("\\nML Feature 유지 확인 완료")
